# MATH-500 Clean Accuracy Run

**What this does:** runs a sample of MATH-500 problems through the pilot model at a large thinking budget and checks accuracy. NOT a drift run. Its only job: prove the answer extractor is trustworthy and that accuracy lands near the known reference (~0.832 for this model on MATH-500), before we build the drift experiment on top.

**Upload into this folder first:** `harness.py` (v0.3.4), `math500_matcher.py`, `math500_worker.py`.

**Key fix in this version:** workers are now pinned to their GPU with an explicit `--device` argument instead of relying on `CUDA_VISIBLE_DEVICES` (which wasn't being inherited on this JupyterHub instance, so every worker piled onto GPU 0 and OOM'd). Run top to bottom.

## 1. Install packages
Let this finish completely before running anything else.

In [ ]:
!pip install -q numpy torch transformers accelerate datasets


## 2. Version check — FIRST, every time
Confirms the on-disk files are the fixed versions before any GPU time. If any line is wrong, fix the upload before continuing.

In [ ]:
import harness, math500_matcher, inspect
print("harness version:", harness.HARNESS_VERSION, "(want 0.3.4)")
print("harness has LaTeX-noise strip:", hasattr(harness, "_strip_latex_noise"), "(want True)")
src = inspect.getsource(math500_matcher)
print("matcher brace-less frac fix:", "_FRAC_BARE" in src, "(want True)")
print("matcher nested frac fix:", "_FRAC_BRACED" in src, "(want True)")

import torch
print("visible GPUs:", torch.cuda.device_count())


## 3. Pick the sample
n=60, fixed seed, large budget so hard items don't truncate.

In [ ]:
import random
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
N = 60
BUDGET = 16384
idxs = list(range(len(ds)))
random.Random(0).shuffle(idxs)
all_idxs = idxs[:N]
print(f"selected {len(all_idxs)} problems, budget {BUDGET} tokens")


## 4. Smoke test — 2 workers on GPUs 1 and 2, 1 item each
Confirms workers launch, land on the RIGHT GPU, and exit cleanly. Note we now pass `--device` explicitly and do NOT set CUDA_VISIBLE_DEVICES. Both should exit 0, and each log should say `using cuda:1` / `using cuda:2`, not cuda:0.

In [ ]:
import subprocess, time

def launch(gpu_id, indices, budget, out):
    # NOTE: no CUDA_VISIBLE_DEVICES. We pin via --device instead, which was
    # the reliable path on this instance. --gpu is just for log labels.
    log = open(f"{out}.log", "w")
    return subprocess.Popen(
        ["python", "math500_worker.py", "--gpu", str(gpu_id), "--device", str(gpu_id),
         "--indices", ",".join(map(str, indices)), "--budget", str(budget), "--out", out],
        stdout=log, stderr=subprocess.STDOUT,
    ), log

test_procs = []
for gpu_id, idx in zip([1, 2], all_idxs[:2]):
    p, log = launch(gpu_id, [idx], BUDGET, f"acc_test_gpu{gpu_id}.json")
    test_procs.append((gpu_id, p, log))
    print(f"launched test on gpu {gpu_id} for idx {idx}")

while any(p.poll() is None for _, p, _ in test_procs):
    time.sleep(10)
for gpu_id, p, log in test_procs:
    print(f"gpu {gpu_id}: exit code {p.returncode}  (0 = good)")

# show each test log so you can confirm it used the right device
for gpu_id, _, _ in test_procs:
    print(f"\n--- acc_test_gpu{gpu_id}.json.log ---")
    print(open(f"acc_test_gpu{gpu_id}.json.log").read())


## 5. Full launch — GPUs 1 through 7
Only run once the smoke test shows exit 0 AND the logs confirm each worker used its own cuda:N, not cuda:0.

In [ ]:
WORKER_GPUS = list(range(1, 8))
shards = [all_idxs[i::len(WORKER_GPUS)] for i in range(len(WORKER_GPUS))]

procs = []
for gpu_id, shard in zip(WORKER_GPUS, shards):
    if not shard:
        continue
    p, log = launch(gpu_id, shard, BUDGET, f"acc_shard_gpu{gpu_id}.json")
    procs.append((gpu_id, p, log))
    print(f"launched gpu {gpu_id} with {len(shard)} items")
print(f"\n{len(procs)} workers running across GPUs {WORKER_GPUS}")


## 6. Check progress
Re-run any time; just reports status.

In [ ]:
done = sum(1 for _, p, _ in procs if p.poll() is not None)
print(f"{done}/{len(procs)} workers finished")
for gpu_id, p, _ in procs:
    print(f"  gpu {gpu_id}: {'done' if p.poll() is not None else 'running'}")


## 7. Merge and report
Run once all workers show done. Reports accuracy, truncation count, and tier breakdown. Because each worker saves after every item, even a crashed worker's completed items are still counted.

In [ ]:
import json
from collections import Counter

all_results = []
for gpu_id, p, log in procs:
    try:
        all_results.extend(json.load(open(f"acc_shard_gpu{gpu_id}.json")))
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"WARNING: no/partial output for gpu {gpu_id}, check acc_shard_gpu{gpu_id}.json.log")

with open("math500_accuracy_run.json", "w") as f:
    json.dump(all_results, f, indent=2)

n = len(all_results)
if n == 0:
    print("No results collected. Check the per-gpu .log files (Section 4 pattern) for errors.")
else:
    n_correct = sum(r["correct"] for r in all_results)
    n_trunc = sum(1 for r in all_results if r["pred"] is None)
    n_fin = n - n_trunc
    print(f"Total items:           {n}")
    print(f"Truncated (pred=None): {n_trunc}")
    print(f"Accuracy (all):        {n_correct}/{n} = {n_correct/n:.3f}")
    if n_fin:
        fc = sum(r['correct'] for r in all_results if r['pred'] is not None)
        print(f"Accuracy (finished):   {fc}/{n_fin} = {fc/n_fin:.3f}")
    print("Reference (Mirage, 8B, MATH-500): ~0.832")
    print("Tier breakdown:", Counter(r['tier'] for r in all_results))


## 8. Hand-check the mismatches
Non-truncation `no_match` rows need a human glance to confirm they're real model errors, not matcher gaps.

In [ ]:
to_check = [r for r in all_results if r["tier"] == "no_match" and r["pred"] is not None]
print(f"{len(to_check)} non-truncation mismatches to hand-check:\n")
for r in to_check:
    print(f"idx={r['math500_idx']} lvl={r.get('level')} subj={r.get('subject')}")
    print(f"   pred: {r['pred']!r}")
    print(f"   gold: {r['gold']!r}\n")
